# MVDet + YOLO pseudo labels

Select `CONFIG["PSEUDO_METHOD"]` (`gaussian` or `disk`) and `CONFIG["ARCH"]` (`resnet18` or `vggt`) in the next cell. The notebook generates the YOLO cache and passes these options to `MVDetBRL/main.py`.

Before running on Kaggle, push the updated source to the branch named in `CONFIG["REPO_BRANCH"]`, or point the URL and branch to another updated checkout. For VGGT, the notebook installs the official code and downloads `facebook/VGGT-1B` unless a local checkpoint is configured.


In [ ]:
from pathlib import Path
from pprint import pprint
import os

CONFIG = {
    # Repository branch containing the MVDet + BRL pseudo-cache pipeline.
    "REPO_URL": "https://github.com/duclld1709/multiview-pedestrian-detection.git",
    "REPO_BRANCH": "pseudoloss_gaussian",
    "REPO_SUBDIR": "MVDetBRL",

    # Leave empty to auto-discover a complete Wildtrack dataset.
    "WILDTRACK_ROOT": "/kaggle/input/datasets/lee735/thesis-dataset-new/Wildtrack/Wildtrack",

    # YOLO is used only to generate the repository's offline JSON cache.
    "YOLO_WEIGHTS": "yolo26x.pt",
    "YOLO_CONFIDENCE": 0.20,
    "YOLO_IMGSZ": 1280,
    "YOLO_DEVICE": "0",

    # All methods are selected in source code; this notebook only passes options.
    "DROP_RATIO": 60,
    "ARCH": "resnet18",  # resnet18 or vggt
    "VGGT_WEIGHTS": "",  # empty: download official facebook/VGGT-1B/model.pt
    "VGGT_INPUT_WIDTH": 518,
    "PSEUDO_METHOD": "gaussian",  # disk or gaussian
    "EPOCHS": 10,
    "BATCH_SIZE": 1,
    "NUM_WORKERS": 4,
    "LR": 0.1,
    "MOMENTUM": 0.5,
    "WEIGHT_DECAY": 0.0005,
    "SEED": 1,
    "LOG_INTERVAL": 10,
    "PSEUDO_LOSS_WEIGHT": 0.01,
    "PSEUDO_CONF_THRESHOLD": 0.20,
    "PSEUDO_SIGMA_M": 0.5,
    "PSEUDO_SUPPRESS_RADIUS_M": 0.5,
}

WORK_ROOT = Path("/kaggle/working")
PROJECT_ROOT = WORK_ROOT / "mvdet_yolo26x_project"
MVDET_ROOT = PROJECT_ROOT / CONFIG["REPO_SUBDIR"]
CACHE_PATH = WORK_ROOT / "pseudo_cache" / "wildtrack_yolo26x.json"
TRAIN_DATA_ROOT = WORK_ROOT / "Wildtrack_runtime"
RESULT_ROOT = WORK_ROOT / "mvdet_yolo26x_results"
os.environ["PYTHONUNBUFFERED"] = "1"
pprint(CONFIG)

In [ ]:
# Install only runtime packages needed by the repository. Keep Kaggle's CUDA PyTorch build.
import json
from collections import deque
import shutil
import subprocess
import sys
import time
import torch

def log(message):
    print(f"[{time.strftime('%Y-%m-%d %H:%M:%S')}] {message}", flush=True)

def run_stream(command, cwd=None, output_file=None):
    command = [str(part) for part in command]
    log("RUN: " + " ".join(command))
    output_handle = open(output_file, "w", encoding="utf-8") if output_file else None
    output_tail = deque(maxlen=80)
    try:
        process = subprocess.Popen(
            command, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, bufsize=1, env={**os.environ, "PYTHONUNBUFFERED": "1"},
        )
        for line in process.stdout:
            print(line, end="", flush=True)
            output_tail.append(line)
            if output_handle:
                output_handle.write(line)
                output_handle.flush()
        return_code = process.wait()
        if return_code:
            location = f" Full output: {output_file}." if output_file else ""
            raise RuntimeError(
                f"Command exited with status {return_code}.{location}\n"
                f"Last output:\n{''.join(output_tail)}"
            )
    finally:
        if output_handle:
            output_handle.close()

def complete_wildtrack(root):
    required = [
        root / "annotations_positions",
        root / "Image_subsets" / "C1",
        root / "Image_subsets" / "C7",
        root / "calibrations" / "intrinsic_zero",
        root / "calibrations" / "extrinsic",
    ]
    if CONFIG["DROP_RATIO"] > 0:
        required.append(root / "drop_annotations" / f"drop_{CONFIG['DROP_RATIO']}" / "annotations_positions")
    return all(path.is_dir() for path in required)

explicit_root = Path(CONFIG["WILDTRACK_ROOT"]) if CONFIG["WILDTRACK_ROOT"] else None
if explicit_root and complete_wildtrack(explicit_root):
    DATA_ROOT = explicit_root
else:
    candidates = []
    for annotations in Path("/kaggle/input").rglob("annotations_positions"):
        candidate = annotations.parent
        if complete_wildtrack(candidate):
            candidates.append(candidate)
    if not candidates:
        raise FileNotFoundError("A complete Wildtrack dataset was not found below /kaggle/input")
    candidates.sort(key=lambda path: ("wildtrack" not in str(path).lower(), len(path.parts)))
    DATA_ROOT = candidates[0]

if not torch.cuda.is_available():
    raise RuntimeError("Enable a GPU accelerator in Kaggle settings")
if shutil.which("git") is None:
    raise RuntimeError("git is unavailable in this runtime")

run_stream([sys.executable, "-m", "pip", "install", "--disable-pip-version-check",
            "kornia==0.7.3", "ultralytics==8.4.69"])
if CONFIG["ARCH"] == "vggt":
    run_stream([sys.executable, "-m", "pip", "install", "huggingface_hub", "einops", "safetensors"])
    run_stream([sys.executable, "-m", "pip", "install", "--no-deps",
                "git+https://github.com/facebookresearch/vggt.git"])
if PROJECT_ROOT.exists():
    shutil.rmtree(PROJECT_ROOT)
run_stream(["git", "clone", "--depth", "1", "--branch", CONFIG["REPO_BRANCH"],
            CONFIG["REPO_URL"], PROJECT_ROOT])

required_repo_files = [
    MVDET_ROOT / "main.py",
    MVDET_ROOT / "generate_pseudo_cache.py",
    MVDET_ROOT / "evaluate_pseudo_cache.py",
]
missing_repo_files = [str(path) for path in required_repo_files if not path.is_file()]
if missing_repo_files:
    raise FileNotFoundError("Repository checkout is incomplete: " + ", ".join(missing_repo_files))
main_source = (MVDET_ROOT / "main.py").read_text(encoding="utf-8")
if "--pseudo_method" not in main_source or "VGGTDetector" not in main_source:
    raise RuntimeError("Checkout lacks source-level Gaussian/VGGT options. Push this branch before running the notebook.")

COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=PROJECT_ROOT, text=True).strip()
# MVDet gốc luôn tạo lại <data_root>/gt.txt. Kaggle input là read-only, nên
# dựng một writable overlay: dữ liệu lớn vẫn nằm nguyên trong /kaggle/input,
# còn gt.txt được ghi trực tiếp dưới /kaggle/working. Không sửa source repo.
if TRAIN_DATA_ROOT.exists() or TRAIN_DATA_ROOT.is_symlink():
    if TRAIN_DATA_ROOT.is_symlink():
        TRAIN_DATA_ROOT.unlink()
    else:
        shutil.rmtree(TRAIN_DATA_ROOT)
TRAIN_DATA_ROOT.mkdir(parents=True)
for source in DATA_ROOT.iterdir():
    if source.name == 'gt.txt':
        continue
    os.symlink(source, TRAIN_DATA_ROOT / source.name, target_is_directory=source.is_dir())

if not complete_wildtrack(TRAIN_DATA_ROOT):
    raise RuntimeError('Writable Wildtrack overlay is incomplete')
log(f"Wildtrack source (read-only): {DATA_ROOT}")
log(f"Wildtrack training overlay: {TRAIN_DATA_ROOT}")
log(f"Repository commit: {COMMIT}")
log(f"GPU: {torch.cuda.get_device_name(0)}")
log(f"Method: MVDet + BRL, pseudo={CONFIG['PSEUDO_METHOD']}, encoder={CONFIG['ARCH']}")

In [ ]:
# Generate the repository-format YOLO26x cache; Gaussian BEV targets are built during training.
CACHE_PATH.parent.mkdir(parents=True, exist_ok=True)
pseudo_generation_log = WORK_ROOT / "yolo26x_pseudo_generation.log"
generate_command = [
    sys.executable, "generate_pseudo_cache.py",
    "--dataset", "wildtrack",
    "--data_root", DATA_ROOT,
    "--output", CACHE_PATH,
    "--weights", CONFIG["YOLO_WEIGHTS"],
    "--confidence", CONFIG["YOLO_CONFIDENCE"],
    "--imgsz", CONFIG["YOLO_IMGSZ"],
    "--device", CONFIG["YOLO_DEVICE"],
]
run_stream(generate_command, cwd=MVDET_ROOT, output_file=pseudo_generation_log)

with CACHE_PATH.open(encoding="utf-8") as cache_file:
    cache = json.load(cache_file)
if cache.get("dataset") != "wildtrack" or not isinstance(cache.get("detections"), dict):
    raise RuntimeError("Generated pseudo cache does not match the MVDet repository schema")
camera_frames = sum(len(cameras) for cameras in cache["detections"].values())
detections = sum(len(rows) for cameras in cache["detections"].values() for rows in cameras.values())
print(f"Cache ready: {CACHE_PATH}")
print(f"frames={len(cache['detections'])}, camera_frames={camera_frames}, detections={detections}")

In [ ]:
# Evaluate with the evaluator already implemented in MVDetBRL.
pseudo_evaluation_log = WORK_ROOT / "yolo26x_pseudo_evaluation.log"
evaluate_command = [
    sys.executable, "evaluate_pseudo_cache.py",
    "--dataset", "wildtrack",
    "--data_root", DATA_ROOT,
    "--cache", CACHE_PATH,
    "--confidence", CONFIG["PSEUDO_CONF_THRESHOLD"],
]
run_stream(evaluate_command, cwd=MVDET_ROOT, output_file=pseudo_evaluation_log)

In [ ]:
# Train through repository options; no notebook monkey patching.
train_command = [
    sys.executable, "main.py",
    "--dataset", "wildtrack",
    "--data_path", TRAIN_DATA_ROOT,
    "--variant", "default",
    "--arch", CONFIG["ARCH"],
    "--pseudo_method", CONFIG["PSEUDO_METHOD"],
    "--loss", "brl",
    "--drop_ratio", CONFIG["DROP_RATIO"],
    "--use_pseudo_labels",
    "--pseudo_cache", CACHE_PATH,
    "--pseudo_loss_weight", CONFIG["PSEUDO_LOSS_WEIGHT"],
    "--pseudo_conf_threshold", CONFIG["PSEUDO_CONF_THRESHOLD"],
    "--pseudo_sigma_m", CONFIG["PSEUDO_SIGMA_M"],
    "--pseudo_suppress_radius_m", CONFIG["PSEUDO_SUPPRESS_RADIUS_M"],
    "--epochs", CONFIG["EPOCHS"],
    "--batch_size", CONFIG["BATCH_SIZE"],
    "--num_workers", CONFIG["NUM_WORKERS"],
    "--lr", CONFIG["LR"],
    "--momentum", CONFIG["MOMENTUM"],
    "--weight_decay", CONFIG["WEIGHT_DECAY"],
    "--seed", CONFIG["SEED"],
    "--log_interval", CONFIG["LOG_INTERVAL"],
]

if CONFIG["ARCH"] == "vggt":
    train_command += ["--vggt_input_width", CONFIG["VGGT_INPUT_WIDTH"]]
    if CONFIG["VGGT_WEIGHTS"]:
        train_command += ["--vggt_weights", CONFIG["VGGT_WEIGHTS"]]

training_driver_log = WORK_ROOT / "mvdet_training_driver.log"
training_error = None
try:
    run_stream(train_command, cwd=MVDET_ROOT, output_file=training_driver_log)
except Exception as error:
    training_error = error
finally:
    log_files = sorted((MVDET_ROOT / "logs").rglob("log.txt"), key=lambda path: path.stat().st_mtime)
    if RESULT_ROOT.exists():
        shutil.rmtree(RESULT_ROOT)
    RESULT_ROOT.mkdir(parents=True)
    if log_files:
        training_log_dir = log_files[-1].parent
        shutil.copytree(training_log_dir, RESULT_ROOT / "training_log")
        print("Gaussian log directory:", training_log_dir)
    else:
        training_log_dir = None
    for artifact in (CACHE_PATH, pseudo_generation_log, pseudo_evaluation_log, training_driver_log):
        if artifact.is_file():
            shutil.copy2(artifact, RESULT_ROOT / artifact.name)
    manifest = {
        "repository": {"url": CONFIG["REPO_URL"], "branch": CONFIG["REPO_BRANCH"], "commit": COMMIT},
        "data_root": str(DATA_ROOT),
        "training_data_overlay": str(TRAIN_DATA_ROOT),
        "method": {
            "mvdet_variant": "default",
            "encoder": CONFIG["ARCH"],
            "mvdet_model": "VGGTDetector" if CONFIG["ARCH"] == "vggt" else "PerspTransDetector",
            "loss": "brl_with_source_pseudo_loss",
            "pseudo_target": CONFIG["PSEUDO_METHOD"],
            "pseudo_weight": "max of YOLO confidence times Gaussian; normalized weighted MSE",
            "pseudo_suppress_radius_m": CONFIG["PSEUDO_SUPPRESS_RADIUS_M"],
            "pseudo_detector": CONFIG["YOLO_WEIGHTS"],
            "puma": False,
            "temporal_fusion": False,
            "tracking": False,
            "pose_keypoints": False,
        },
        "config": CONFIG,
        "train_command": [str(part) for part in train_command],
        "training_log_dir": str(training_log_dir) if training_log_dir else None,
    }
    (RESULT_ROOT / "run_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    archive = shutil.make_archive(str(WORK_ROOT / "mvdet_yolo26x_results"), "zip", root_dir=RESULT_ROOT)
    print("Results archive:", archive)
if training_error is not None:
    raise training_error

In [ ]:
# Inspect artifacts from the notebook Gaussian pseudo run.
for path in sorted(RESULT_ROOT.rglob("*")):
    if path.is_file():
        print(path.relative_to(RESULT_ROOT), f"({path.stat().st_size / 1024 / 1024:.2f} MiB)")

training_log = RESULT_ROOT / "training_log" / "log.txt"
if training_log.is_file():
    print("\n--- Tail of training log.txt ---")
    print("".join(training_log.read_text(encoding="utf-8").splitlines(keepends=True)[-40:]))